In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import poisson

# Load Relevant Data

In [2]:
import pickle
with open('background_sr1_df_ac_model_ML_pax6.11.1_lones1_pax6.11.1_hax2.5.0_lax1.7.2_file0.pkl', 'rb') as f:
    df_CEvNS = pickle.load(f) 

In [3]:
df_s2only = pd.read_csv("s2only_events_pass_cuts.csv")
#https://github.com/XENON1T/s2only/blob/master/analyses/datapoints/events_pass_cuts.csv - SR0 absent

In [4]:
df_NR = pd.read_hdf('NR_events.hdf5','data')
#https://github.com/XENON1T/SR1Results/blob/master/Plots/unblind_basic.ipynb

# Event rate

In [5]:
CEvNS_events = np.shape(df_CEvNS)[0]
CEvNS_events

6

In [6]:
CEvNS_obs_days = 246.7 #https://xe1t-wiki.lngs.infn.it/doku.php?id=xenon:xenon1t:analysis:cevns_b8

In [7]:
s2only_events = np.shape(df_s2only)[0]
s2only_events

170

In [8]:
s2only_obs_days = 258.2 #https://arxiv.org/pdf/1907.11485.pdf

In [9]:
NR_events = np.shape(df_NR)[0]
NR_events

14

In [10]:
NR_obs_days = 278.8 #https://arxiv.org/pdf/1805.12562.pdf

In [11]:
channels = ['CEvNS', 's2only', 'NR']

data = {'Event-type': channels,
        'event count': [CEvNS_events, s2only_events, NR_events],
        'Observation days' : [CEvNS_obs_days, s2only_obs_days, NR_obs_days]
        }
Table = pd.DataFrame(data)
Table['event rate (events/day)'] = Table['event count']/Table['Observation days']
Table['event rate (events/1000 seconds)'] = Table['event rate (events/day)']*1000/(24*60*60)
Table['P_no_obs_GW'] = poisson.cdf(0, Table['event rate (events/1000 seconds)'])
Table

,Event-type,Observation days,event count,event rate (events/day),event rate (events/1000 seconds),P_no_obs_GW
0,CEvNS,246.7,6,0.024321,0.000281,0.999719
1,s2only,258.2,170,0.658404,0.007620,0.992409
2,NR,278.8,14,0.050215,0.000581,0.999419


# Sideband time regions

In [12]:
# intialise data of lists.

from datetime import datetime, timezone, timedelta

GW_8_23_datetime = datetime(2017, 8, 23, 13, 13, 58, tzinfo=timezone.utc)
GW_8_18_datetime = datetime(2017, 8, 18, 2, 25, 9, tzinfo=timezone.utc)
GW_8_17_datetime = datetime(2017, 8, 17, 12, 41, 4, tzinfo=timezone.utc)
GW_7_29_datetime = datetime(2017, 7, 29, 18, 56, 29, tzinfo=timezone.utc)
GW_1_04_datetime = datetime(2017, 1, 4, 10, 11, 58, tzinfo=timezone.utc)

GW_1_04_time = GW_1_04_datetime.timestamp()
GW_7_29_time = GW_7_29_datetime.timestamp()
GW_8_17_time = GW_8_17_datetime.timestamp()
GW_8_18_time = GW_8_18_datetime.timestamp()
GW_8_23_time = GW_8_23_datetime.timestamp()

GW = ['GW170104', 'GW170729', 'GW170817', 'GW170818', 'GW170823']

data = {'Event': GW,
        'Event_time': [GW_1_04_time, GW_7_29_time, GW_8_17_time, GW_8_18_time, GW_8_23_time],
        }
Data = pd.DataFrame(data)
Data['n_CEvNS'] = 0
Data['n_s2only'] = 0
Data['n_NR'] = 0

Data

,Event,Event_time,n_CEvNS,n_s2only,n_NR
0,GW170104,1.483525e+09,0,0,0
1,GW170729,1.501355e+09,0,0,0
2,GW170817,1.502974e+09,0,0,0
3,GW170818,1.503023e+09,0,0,0
4,GW170823,1.503494e+09,0,0,0


### Observed Events

In [13]:
for j in range(0, np.shape(Data['Event_time'])[0]):
    for i in range(0, np.shape(df_CEvNS['event_time'])[0]):
        id = df_CEvNS.index[i]
        delay = Data['Event_time'][j] - df_CEvNS['event_time'][id]*1e-9 
        if delay > 500 and delay<1500:
            Data['n_CEvNS'][j] = Data['n_CEvNS'][j] + 1
            
    for i in range(0, np.shape(df_s2only['event_time'])[0]):
        id = df_s2only.index[i]
        delay = Data['Event_time'][j] - df_s2only['event_time'][id]*1e-9 
        if delay > 500 and delay<1500:
            Data['n_s2only'][j] = Data['n_s2only'][j] + 1
            
    for i in range(0, np.shape(df_NR['event_time'])[0]):
        id = df_NR.index[i]
        delay = Data['Event_time'][j] - df_NR['event_time'][id]*1e-9 
        if delay > 500 and delay<1500:
            Data['n_NR'][j] = Data['n_NR'][j] + 1
            
Data

,Event,Event_time,n_CEvNS,n_s2only,n_NR
0,GW170104,1.483525e+09,0,0,0
1,GW170729,1.501355e+09,0,0,0
2,GW170817,1.502974e+09,0,0,0
3,GW170818,1.503023e+09,0,0,0
4,GW170823,1.503494e+09,0,0,0


# Time region of interest

In [13]:
for j in range(0, np.shape(Data['Event_time'])[0]):
    for i in range(0, np.shape(df_CEvNS['event_time'])[0]):
        id = df_CEvNS.index[i]
        delay = abs(Data['Event_time'][j] - df_CEvNS['event_time'][id])*1e-9 
        if delay<500:
            Data['n_CEvNS'][j] = Data['n_CEvNS'][j] + 1
            
    for i in range(0, np.shape(df_s2only['event_time'])[0]):
        id = df_s2only.index[i]
        delay = abs(Data['Event_time'][j] - df_s2only['event_time'][id])*1e-9 
        if delay<500:
            Data['n_s2only'][j] = Data['n_s2only'][j] + 1
            
    for i in range(0, np.shape(df_NR['event_time'])[0]):
        id = df_NR.index[i]
        delay = abs(Data['Event_time'][j] - df_NR['event_time'][id])*1e-9 
        if delay<500:
            Data['n_NR'][j] = Data['n_NR'][j] + 1
            
Data

,Event,Event_time,n_CEvNS,n_s2only,n_NR
0,GW170104,1.483525e+09,0,0,0
1,GW170729,1.501355e+09,0,0,0
2,GW170817,1.502974e+09,0,0,0
3,GW170818,1.503023e+09,0,0,0
4,GW170823,1.503494e+09,0,0,0


### Finding livetime

In [14]:
import hax
pax_version = '6.10.1'

hax.init(experiment='XENON1T',
         pax_version_policy=pax_version,
         minitree_paths=['scratch-midway2/miniforest/pax_v'+ pax_version,
                         '/project2/lgrandi/xenon1t/minitrees/pax_v'+ pax_version,
                         '/dali/lgrandi/xenon1t/minitrees/pax_v'+ pax_version,
                        ],
         main_data_paths=['/dali/lgrandi/xenon1t/processed/pax_v'+ pax_version,
                          '/project2/lgrandi/xenon1t/processed/pax_v'+ pax_version,
                         ],)

dsets = hax.runs.datasets

dsets = hax.runs.tags_selection(dsets, include=['sciencerun0','sciencerun1','GW'],
                            exclude=['NoMV','Flash','PMTramping','readoutbug','flash','bad', 'messy', '*trip', '*quake','test','NG','MVoff'])

dsets = dsets[
    (dsets['location'] != "")
    & (dsets.source__type == 'none')
]

dataset_start = []
dataset_end = []
for i in range(0, np.shape(dsets)[0]):
    id = dsets.index[i]
    dataset_start.append(dsets.start[id].timestamp())
    dataset_end.append(dsets.end[id].timestamp())


In [15]:
Data['start time'] = ''
Data['end time'] = ''

for j in range(0, np.shape(Data['Event_time'])[0]):
    Data['start time'][j] = []
    Data['end time'][j] = []
    for i in range(0, np.size(dataset_start)):
        delay_s = Data['Event_time'][j] - dataset_start[i]
        if delay_s > 500 and delay_s<1500:
            Data['start time'][j].append(dataset_start[i])

        delay_e = Data['Event_time'][j] - dataset_end[i]
        if delay_e > 500 and delay_e<1500:
            Data['end time'][j].append(dataset_end[i])

Data

/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy


,Event,Event_time,n_CEvNS,n_s2only,n_NR,start time,end time
0,GW170104,1.483525e+09,0,0,0,[],[]
1,GW170729,1.501355e+09,0,0,0,[],[]
2,GW170817,1.502974e+09,0,0,0,[],[]
3,GW170818,1.503023e+09,0,0,0,[1503022261.0],[1503022254.0]
4,GW170823,1.503494e+09,0,0,0,[],[]


In [20]:
Data['missing_time'] = 0
for i in range(0, 5):
    for j in range(0, np.shape(Data['start time'][i])[0]):
        Data['missing_time'][i] = Data['missing_time'][i] + Data['start time'][i][j] - Data['end time'][i][j]
Data['livetime'] = 1000 - Data['missing_time']
Data

/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy


,Event,Event_time,n_CEvNS,n_s2only,n_NR,start time,end time,missing_time,livetime
0,GW170104,1.483525e+09,0,0,0,[],[],0,1000
1,GW170729,1.501355e+09,0,0,0,[],[],0,1000
2,GW170817,1.502974e+09,0,0,0,[],[],0,1000
3,GW170818,1.503023e+09,0,0,0,[1503022261.0],[1503022254.0],7,993
4,GW170823,1.503494e+09,0,0,0,[],[],0,1000


In [21]:
Data['Expected bkgd CEvNS events'] = Table['event rate (events/1000 seconds)'][0]*Data['livetime']/1000
Data['Expected bkgd s2only events'] = Table['event rate (events/1000 seconds)'][1]*Data['livetime']/1000
Data['Expected bkgd NR events'] = Table['event rate (events/1000 seconds)'][2]*Data['livetime']/1000
Data

,Event,Event_time,n_CEvNS,n_s2only,n_NR,start time,end time,missing_time,livetime,Expected bkgd CEvNS events,Expected bkgd s2only events,Expected bkgd NR events
0,GW170104,1.483525e+09,0,0,0,[],[],0,1000,0.000281,0.007620,0.000581
1,GW170729,1.501355e+09,0,0,0,[],[],0,1000,0.000281,0.007620,0.000581
2,GW170817,1.502974e+09,0,0,0,[],[],0,1000,0.000281,0.007620,0.000581
3,GW170818,1.503023e+09,0,0,0,[1503022261.0],[1503022254.0],7,993,0.000280,0.007567,0.000577
4,GW170823,1.503494e+09,0,0,0,[],[],0,1000,0.000281,0.007620,0.000581
